[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/habin-k/F1-simulator/blob/main/notebooks/02_telemetry_track_map.ipynb)

# F1 SIM 02 - Telemetry and Track Map

This notebook loads one session from FastF1 (public F1 timing data), picks the fastest lap,
plots its telemetry against distance, draws a speed-coloured track map, and writes the lap
to a small CSV that later notebooks use as input.

- Session: Monza 2025 Qualifying (change `YEAR`, `GP`, `SESSION` in the setup cell)
- Needs internet access to the FastF1 data servers on the first run. Data is cached in `cache/`.
- Units: `Distance` in m, `X`/`Y` in 1/10 m, `Speed` in km/h.


## 0. Install

In [ ]:
!pip install -q fastf1==3.9.2

## 1. Setup

On Colab this notebook clones the public repository to `/content/F1-simulator` and imports
the loader functions from `src/f1sim/data.py`. When run from a local clone, it uses that clone.

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

REPO = Path("..").resolve()
if not (REPO / "src" / "f1sim" / "data.py").exists():
    REPO = Path("/content/F1-simulator")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             "https://github.com/habin-k/F1-simulator.git", str(REPO)],
            check=True,
        )
sys.path.insert(0, str(REPO / "src"))

from f1sim.data import fastest_lap_telemetry, load_session, sample_filename, save_lap_csv  # noqa: E402

# Values to change
YEAR = 2025
GP = "Monza"
SESSION = "Q"  # Q: qualifying, R: race
CACHE_DIR = "cache"

def fmt_laptime(td):
    minutes, seconds = divmod(td.total_seconds(), 60)
    return f"{int(minutes)}:{seconds:06.3f}"


## 2. Load the session

In [ ]:
session = load_session(YEAR, GP, SESSION, cache_dir=CACHE_DIR, telemetry=True, weather=False, messages=False)
print(session.event["EventName"], "-", session.name)

## 3. Fastest lap and its telemetry

In [ ]:
lap, tel = fastest_lap_telemetry(session)
print("Driver:", lap["Driver"], "| Lap time:", fmt_laptime(lap["LapTime"]))
print("Samples:", len(tel))
tel[["Distance", "Speed", "Throttle", "Brake", "nGear", "X", "Y"]].head()

## 4. Speed, throttle, and brake versus distance

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

axes[0].plot(tel["Distance"], tel["Speed"], color="tab:blue")
axes[0].set_ylabel("Speed (km/h)")

axes[1].plot(tel["Distance"], tel["Throttle"], color="tab:green")
axes[1].set_ylabel("Throttle (%)")

axes[2].plot(tel["Distance"], tel["Brake"].astype(int), color="tab:red")
axes[2].set_ylabel("Brake (0/1)")
axes[2].set_yticks([0, 1])
axes[2].set_xlabel("Distance (m)")

fig.suptitle(f"{session.event['EventName']} {YEAR} {session.name} - {lap['Driver']}")
fig.tight_layout()
plt.show()

## 5. Speed-coloured track map

In [ ]:
# X/Y are in 1/10 m. Convert to m so the axes are readable.
x = tel["X"].to_numpy() / 10
y = tel["Y"].to_numpy() / 10
speed = tel["Speed"].to_numpy()

# Join neighbouring points into line segments and colour each segment by speed.
points = np.column_stack([x, y]).reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

fig, ax = plt.subplots(figsize=(7, 7))
lc = LineCollection(segments, cmap="plasma", linewidth=4)
lc.set_array(speed[:-1])
ax.add_collection(lc)

# Pad each axis by 5% of its range so the track is not clipped.
pad_x = 0.05 * (x.max() - x.min())
pad_y = 0.05 * (y.max() - y.min())
ax.set_xlim(x.min() - pad_x, x.max() + pad_x)
ax.set_ylim(y.min() - pad_y, y.max() + pad_y)
ax.set_aspect("equal")
ax.axis("off")
fig.colorbar(lc, ax=ax, shrink=0.7, label="Speed (km/h)")
ax.set_title(f"{session.event['EventName']} {YEAR} - {lap['Driver']} fastest lap")
plt.show()

## 6. Save the lap for the next notebooks

The CSV goes to `data/sample/` in the repository. On Colab it is written to
`/content/F1-simulator/data/sample/`, and you can download it from the Files panel.

In [ ]:
out_dir = REPO / "data" / "sample"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / sample_filename(GP, YEAR, SESSION, "lap_telemetry")
rows = save_lap_csv(tel, out_path)
print("Saved", rows, "rows to", out_path)